In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
from scipy.signal import find_peaks

In [ ]:
data_x_spectrum = np.load("x.npz")


In [ ]:
frequency = data_x_spectrum['frequency']
amplitude = data_x_spectrum['amplitude']

In [ ]:
import remove_edge_and_params as reap
import peak_finder_and_test_plot as pfat
import build_clusters_and_peak_spaces as bcps
import normalization_and_spectra as normspec
import crosscorrelation_and_template_matching as crosstemp
import grouping
import first_harmonic as fh
import continuous_spectrum_and_analysis as csa

In [ ]:
frequency, amplitude = reap.remove_edge_artifacts(
    frequency,
    amplitude,
    mean_window=1001,
    gradient_window=501,
    gradient_threshold_factor=0.15,
    amplitude_threshold_factor=20,
    min_region=500,
    symmetric_edges=True,
    debug=True
)

In [ ]:
params = reap.estimate_spectrum_parameters(
    frequency,
    expected_frev=2.00e6,
    frev_tolerance=0.10e6
)

In [ ]:
# optional : Plot of the spectrum 

# import plotly.graph_objects
# fig = plotly.graph_objects.Figure()
# fig.add_trace(plotly.graph_objects.Scatter(x=frequency,
#     y=amplitude,
#     mode="lines",
#     name="Spectrum"
# ))
# fig.update_layout(
#     title="Interactive Spectrum",
#     xaxis_title="frequency",
#     yaxis_title="amplitude"
# )


# fig.show()


In [ ]:
peaks, resonance_mask, activity = pfat.find_peaks_adaptive(
        frequency,
        amplitude,
        prominence_resonance=params["prominence_resonance"],   
        distance_resonance=params["distance_resonance"],
        threshold_background=params["threshold_background"],
        merge_background_points=params["merge_background_points"],
        activity_window=300,
        activity_fraction=0.40
    )

In [ ]:
# Plot in oder to see if the values are set correctly 
pfat.plot_spectrum_with_peaks_plotly(
    frequency,
    amplitude,
    peaks
)

In [ ]:
peaks, resonance_mask, activity = pfat.find_peaks_adaptive(
        frequency,
        amplitude,
        prominence_resonance=params["prominence_resonance"],
        distance_resonance=params["distance_resonance"],
        threshold_background=params["threshold_background"],
        merge_background_points=params["merge_background_points"],
        activity_window=300,
        activity_fraction=0.40
    )


print("Detected peaks:", len(peaks))


(
    peaks,
    peak_freqs,
    peak_amps,
    peak_activity
) = bcps.build_peak_space(
    frequency,
    amplitude,
    peaks,
    activity
)


peak_to_group = {}



clusters = bcps.cluster_peaks_by_frequency(
    peaks=np.arange(len(peak_freqs)),
    frequency=peak_freqs,
    amplitude=peak_amps,
    peak_to_group=peak_to_group,
    cluster_gap_threshold=0.5e5,
    amp_ratio_threshold=0.6,
    valley_threshold=0.4
)



# zurück zu Originalindizes
clusters_original = [
    peaks[c]
    for c in clusters
]


print()
print("================================")
print("Cluster result")
print("================================")
print("Number of clusters:", len(clusters_original))
print(
    "Cluster sizes:",
    [len(c) for c in clusters_original]
)

In [ ]:
print("Number of Peaks:", len(peaks))
print("Number of Clusters:", len(clusters))

print(
    "Peaks in Clusters:",
    sum(len(c) for c in clusters)
)

In [ ]:
raw_peak_results = bcps.extract_peak_segments_raw(
    frequency,
    amplitude,
    clusters_original,
    peak_to_group
)

In [ ]:
# Shape Spectrum
spec_orig = normspec.build_shape_spectrum_from_clusters(
    raw_peak_results,
    frequency
)
# not normalised spectrum
spec_raw = normspec.build_raw_spectrum_from_clusters(
    raw_peak_results,
    frequency
)

selected_peaks = np.array([
    np.argmax(r["y"]) + np.searchsorted(frequency, r["x"][0])
    for r in raw_peak_results
])
normspec.plot_spectrum_filtered(
    frequency,
    spec_orig,
    selected_peaks
)

In [ ]:
# Debug

print("raw peaks max:",
      max(np.max(r["y"]) for r in raw_peak_results))

print("spec_raw max:",
      np.max(spec_raw))

print("spec_orig max:",
      np.max(spec_orig))

In [ ]:

# Build templates from remaining peaks

templates = crosstemp.compute_cluster_templates(
    raw_peak_results
)


cluster_ids, sim_matrix = crosstemp.compute_similarity_matrix(
    templates
)


cluster_properties = crosstemp.get_cluster_properties(
    raw_peak_results,
    frequency,
    activity=activity,
    resonance_mask=resonance_mask
)

# Cluster properties

cluster_positions = {
    cid: data["position"]
    for cid, data in cluster_properties.items()
}


cluster_amplitudes = {
    cid: data["amplitude"]
    for cid, data in cluster_properties.items()
}


cluster_activity = {
    cid: data["activity"]
    for cid, data in cluster_properties.items()
}


valid_cluster_ids = [
    cid
    for cid in cluster_ids
    if cid in cluster_positions
    and cid in cluster_amplitudes
    and cid in cluster_activity
]


print("=" * 70)
print("GROUPING AFTER SHAPE REMOVAL")
print("=" * 70)

print(
    "Remaining clusters:",
    valid_cluster_ids
)

# PERIODIC GROUPING

groups = grouping.group_clusters_strict_periodic(
    sim_matrix,
    valid_cluster_ids,
    cluster_positions,
    cluster_amplitudes,
    cluster_activity,
    freq_min=params["freq_min"],
    freq_max=params["freq_max"],
    sim_threshold=0.6,
    min_repeats=params["min_repeats"],
    max_repeats=params["max_repeats"],
    harmonic_tolerance=5e4,
    max_spacing_error=5e4,
    max_harmonic_deviation=0.05
)

# SCORE GROUPS

groups = grouping.score_periodic_groups(
    groups,
    cluster_amplitudes,
    cluster_activity,
    valid_cluster_ids,
    sim_matrix
)

# PRINT RESULTS

print("\nDetected periodic groups\n")


for i, g in enumerate(groups):

    print(f"Group {i+1}")

    print(
        f"clusters = {g['clusters']}"
    )

    print(
        f"harmonics = {g['harmonics']}"
    )

    print(
        f"f_rev = {g['harmonic_spacing']/1e6:.6f} MHz"
    )

    print(
        f"fit error = {g['fit_error']/1e3:.2f} kHz"
    )

    print(
        f"score = {g['score']:.3f}"
    )

    print()


In [ ]:
for i, g in enumerate(groups):

    print(f"\n{'='*60}")
    print(f"Group {i+1}")
    print(f"{'='*60}")

    print(
        f"Recovered f_rev : {g['harmonic_spacing']/1e6:.6f} MHz"
    )

    print(
        f"Fit error        : {g['fit_error']/1e3:.2f} kHz"
    )

    print(
        f"Harmonics        : {g['harmonics']}"
    )

    print("\nClusters:")

    for cid, h in zip(g["clusters"], g["harmonics"]):

        print(
            f"  H={h:2d} | "
            f"Cluster {cid:2d} | "
            f"{cluster_positions[cid]/1e6:.6f} MHz | "
            f"Amp = {cluster_amplitudes[cid]:.3f}"
        )

In [ ]:
groups_final = grouping.select_final_groups(
    groups
)

In [ ]:
import numpy as np
import plotly.graph_objects as go


def plot_grouped_clusters_clean(
    frequency,
    spectrum,
    raw_peak_results,
    groups
):

    fig = go.Figure()

    colors = [
        "red", "blue", "green", "orange",
        "purple", "cyan", "magenta", "gold"
    ]

    fig.add_trace(go.Scatter(
        x=frequency,
        y=spectrum,
        mode="lines",
        line=dict(color="lightgray", width=2),
        opacity=0.5,
        name="Spectrum"
    ))

    for gid, group_data in enumerate(groups):

        group = group_data["clusters"]

        color = colors[gid % len(colors)]

        legend_added = False

        for cid in group:

            cluster_results = [
                r for r in raw_peak_results
                if r["cluster_id"] == cid
            ]

            for r in cluster_results:

                left, right = r["window"]

                baseline = 0.005 * np.max(spectrum)

                l = left

                while l > 0 and spectrum[l] > baseline:
                    l -= 1

                if l > 0:
                    l -= 1

                r_idx = right

                while (
                    r_idx < len(spectrum)-1
                    and spectrum[r_idx] > baseline
                ):
                    r_idx += 1

                if r_idx < len(spectrum)-1:
                    r_idx += 1

                x = frequency[l:r_idx]
                y = spectrum[l:r_idx]

                fig.add_trace(go.Scatter(
                    x=x,
                    y=y,
                    mode="lines",
                    line=dict(
                        color=color,
                        width=2
                    ),
                    opacity=0.35,
                    name=f"Group {gid}"
                    if not legend_added
                    else None,
                    showlegend=not legend_added
                ))

                legend_added = True
    
    for gid, group_data in enumerate(groups):

        group = group_data["clusters"]

        color = colors[gid % len(colors)]
        px = []
        py = []

        for cid in group:

            cluster_results = [
                r for r in raw_peak_results
                if r["cluster_id"] == cid
            ]

            for r in cluster_results:

                peak_x = r["peak_position"]

                peak_y = np.interp(
                    peak_x,
                    frequency,
                    spectrum
                )

                px.append(peak_x)
                py.append(peak_y)

        fig.add_trace(go.Scatter(
            x=px,
            y=py,
            mode="markers",
            marker=dict(
                size=8,
                color=color
            ),
            showlegend=False
        ))

    fig.update_layout(
        title="Template-Matching Groups",
        xaxis_title="Frequency (Hz)",
        yaxis_title="Normalized Amplitude",
        template="plotly_white",
        hovermode="closest"
    )

    fig.show()

In [ ]:
plot_grouped_clusters_clean(
    frequency,
    spec_orig,
    raw_peak_results,
    groups_final
)

In [ ]:
periodicity_results = fh.analyze_group_periodicity(
    groups_final,
    cluster_positions
)

In [ ]:
peak_families = fh.build_peak_families_robust(
    groups_final,
    raw_peak_results
)

In [ ]:
display(
    peak_families.sort_values(
        [
            "group_id",
            "family_id",
            "f_peak"
        ]
    )
    [
        [
            "family_id",
            "group_id",
            "cluster_id",
            "peak_index",
            "f_peak",
            "harmonic",
            "fundamental",
            "folded_frequency"
        ]
    ]
)

In [ ]:
fh.plot_mean_peak_families_styled(
    peak_families,
    frequency,
    spec_orig,
    n_points=4000,
    distance=10,             
    prominence_factor=0.03 
)

In [ ]:
#optional: nice plot of the first harmonic with shape
# FIGURE

fig, ax = plt.subplots(
    figsize=(12, 7)
)

# COLORS

colors = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "cyan",
    "magenta",
    "gold",
    "brown",
    "pink",
    "olive"
]


shown = set()

# BASELINE

baseline = (
    0.005 *
    np.max(spec_orig)
)

# PEAK TABLE

all_rows = []


# PEAK-TICKS

peak_positions = []
peak_labels = []

# COMPRESSION

gap_factor = 0.000005

min_gap = 0.0


family_curves = []


for fam_id, fam_df in peak_families.groupby(
    "family_id"
):

    curves = []


    for _, row in fam_df.iterrows():

        left, right = row["window"]

        h = row["harmonic"]

        if (
            h <= 0
            or np.isnan(h)
        ):
            continue

        l = int(left)


        while (
            l > 0
            and spec_orig[l] > baseline
        ):

            l -= 1


        if l > 0:

            l -= 1

        r_idx = int(right)


        while (
            r_idx < len(spec_orig) - 1
            and spec_orig[r_idx] > baseline
        ):

            r_idx += 1


        if r_idx < len(spec_orig) - 1:

            r_idx += 1

        x = frequency[l:r_idx]

        y = np.maximum(
            spec_orig[l:r_idx],
            0
        )


        if len(x) < 2:

            continue


        x_folded = (
            x / h
        )


        curves.append(
            (
                x_folded,
                y
            )
        )

    if len(curves) == 0:

        continue

    xmin = max(
        np.min(x)
        for x, _ in curves
    )


    xmax = min(
        np.max(x)
        for x, _ in curves
    )


    if xmax <= xmin:

        continue


    x_common = np.linspace(
        xmin,
        xmax,
        400
    )


    Y = []


    for x_folded, y in curves:

        y_interp = np.interp(
            x_common,
            x_folded,
            y
        )


        Y.append(
            y_interp
        )

    y_mean = np.maximum(
        np.nanmean(
            Y,
            axis=0
        ),
        0
    )

    if np.max(y_mean) > 0:

        raw_peaks, _ = find_peaks(
            y_mean,
            prominence=(
                0.02 *
                np.max(y_mean)
            ),
            distance=5
        )

    else:

        raw_peaks = []


    if len(raw_peaks) == 0:

        peaks = np.array(
            [np.nanargmax(y_mean)],
            dtype=int
        )

    else:

        peaks = np.asarray(
            raw_peaks,
            dtype=int
        )


    for p in peaks:

        original_frequency = (
            x_common[p]
        )


        all_rows.append({

            "family_id":
                fam_id,

            "peak_position":
                original_frequency,

            "peak_height":
                y_mean[p]

        })

    family_curves.append({

        "family_id":
            fam_id,

        "x":
            x_common,

        "y":
            y_mean,

        "peaks":
            peaks

    })


family_curves.sort(
    key=lambda d: d["x"][0]
)

if len(family_curves) > 0:

    
    family_curves[0]["x_plot"] = (
        family_curves[0]["x"].copy()
    )

    for i in range(
        1,
        len(family_curves)
    ):

        previous = (
            family_curves[i - 1]
        )

        current = (
            family_curves[i]
        )

        original_gap = (
            current["x"][0]
            -
            previous["x"][-1]
        )

        if original_gap <= 0:

            new_start = (
                previous["x_plot"][-1]
            )

        else:

            compressed_gap = (
                original_gap *
                gap_factor
            )


            compressed_gap = max(
                compressed_gap,
                min_gap
            )


            new_start = (
                previous["x_plot"][-1]
                +
                compressed_gap
            )

        offset = (
            new_start
            -
            current["x"][0]
        )

        current["x_plot"] = (
            current["x"]
            +
            offset
        )

# PLOT

for data in family_curves:

    fam_id = data["family_id"]

    x_original = data["x"]

    x_plot = data["x_plot"]

    y = data["y"]

    peaks = data["peaks"]

    color = colors[
        int(fam_id) %
        len(colors)
    ]

    x_closed = np.concatenate(
        [
            [x_plot[0]],
            x_plot,
            [x_plot[-1]]
        ]
    )


    y_closed = np.concatenate(
        [
            [0.0],
            y,
            [0.0]
        ]
    )

    ax.plot(
        x_closed / 1e6,
        y_closed,
        color=color,
        linewidth=2,
        label=(
            f"Group {fam_id+1}"
            if fam_id not in shown
            else None
        )
    )


    shown.add(
        fam_id
    )


    ax.scatter(
        x_plot[peaks] / 1e6,
        y[peaks],
        s=55,
        color=color,
        zorder=3
    )

    for p in peaks:

        plot_position = (
            x_plot[p] / 1e6
        )


        original_position = (
            x_original[p] / 1e6
        )


        peak_positions.append(
            plot_position
        )


        peak_labels.append(
            f"{original_position:.6f}"
        )

# PEAK TABLE

peak_table = pd.DataFrame(
    all_rows
)


print(
    "\n===== PEAK TABLE ====="
)

print(
    peak_table
)

for i in range(
    len(family_curves) - 1
):

    left_family = (
        family_curves[i]
    )

    right_family = (
        family_curves[i + 1]
    )

    x_left = (
        left_family["x_plot"][-1]
        / 1e6
    )

    x_right = (
        right_family["x_plot"][0]
        / 1e6
    )

    if x_right <= x_left:

        continue


    x_break = (
        x_left +
        x_right
    ) / 2


    xlim = ax.get_xlim()

    x_width = (
        xlim[1] -
        xlim[0]
    )


    break_width = (
        0.008 *
        x_width
    )


    break_height = (
        0.025 *
        ax.get_ylim()[1]
    )

    ax.plot(
        [
            x_break - break_width,
            x_break - break_width / 3
        ],
        [
            0,
            break_height
        ],
        color="black",
        linewidth=1.5,
        clip_on=False
    )


    ax.plot(
        [
            x_break + break_width / 3,
            x_break + break_width
        ],
        [
            0,
            break_height
        ],
        color="black",
        linewidth=1.5,
        clip_on=False
    )


ax.set_xlabel(
    "Folded frequency [MHz]"
)

ax.set_ylabel(
    "Amplitude"
)


ax.set_ylim(
    bottom=0
)

ax.ticklabel_format(
    axis="x",
    style="plain",
    useOffset=False
)

if len(peak_positions) > 0:

    sorted_peaks = sorted(
        zip(
            peak_positions,
            peak_labels
        ),
        key=lambda item: item[0]
    )


    tick_positions = []
    tick_labels = []

    min_tick_distance = 1e-8


    for position, label in sorted_peaks:

        if len(tick_positions) == 0:

            tick_positions.append(
                position
            )

            tick_labels.append(
                label
            )

        else:

            if (
                abs(
                    position -
                    tick_positions[-1]
                )
                >
                min_tick_distance
            ):

                tick_positions.append(
                    position
                )

                tick_labels.append(
                    label
                )


    ax.set_xticks(
        tick_positions
    )


    ax.set_xticklabels(
        tick_labels,
        rotation=45,
        ha="right",
        fontsize=8
    )

ax.grid(
    alpha=0.2
)

ax.legend(
    fontsize=8,
    loc="upper left",
    bbox_to_anchor=(1.02, 1.0),
    borderaxespad=0
)


plt.tight_layout(
    rect=[
        0,
        0,
        0.82,
        1
    ]
)
# plt.savefig(
#     "first_harmonic.pdf",
#     bbox_inches="tight"
# )

plt.show()

In [ ]:
print("\n====================")
print("SPEC_ORIG")
print("====================")

x_cont_orig, spec_cont_orig = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_orig,
    n_points=600000
)

In [ ]:
x_cont_x, spec_cont_x = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_orig,
    n_points=60000,
    baseline_floor=1e-12
)

np.savez(
    "continuous_spectrum_x.npz",
    frequency=x_cont_x,
    spectrum=spec_cont_x
)

In [ ]:
print("\n====================")
print("SPEC_RAW")
print("====================")

x_cont_raw_x, spec_cont_raw_x = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_raw,
    n_points=600000
)

In [ ]:
x_cont_raw_x, spectrum_cont_raw_x = csa.build_continuous_spectrum_from_mean_families(
    peak_families,
    frequency,
    spec_raw,
    n_points = 600000
)

np.savez(
    "continuous_spectrum_raw_x.npz",
    frequency=x_cont_raw_x,
    spectrum=spectrum_cont_raw_x
)

In [ ]:
# nearest neightbor analysis 
df = peak_families.copy()

fundamentals = df.sort_values("f_peak").reset_index(drop=True)

freqs = fundamentals["f_peak"].values

print("\n===== PEAK POSITIONS (SORTED) =====")

display(
    fundamentals[
        [
            "family_id",
            "cluster_id",
            "f_peak",
            "harmonic",
            "fundamental"
        ]
    ]
)

#  NEAREST NEIGHBORS

diff_rows = []

for i in range(len(fundamentals) - 1):

    j = i + 1

    f1 = freqs[i]
    f2 = freqs[j]

    delta_f = f2 - f1

    diff_rows.append({
        "family_1": fundamentals["family_id"].iloc[i],
        "family_2": fundamentals["family_id"].iloc[j],

        "cluster_1": fundamentals["cluster_id"].iloc[i],
        "cluster_2": fundamentals["cluster_id"].iloc[j],

        "f1_Hz": f1,
        "f2_Hz": f2,

        "delta_f_Hz": delta_f,
        "relative_df": delta_f / f1 if f1 != 0 else np.nan
    })

df_diff = pd.DataFrame(diff_rows)

df_diff = df_diff.sort_values("delta_f_Hz").reset_index(drop=True)

print("\n===== NEAREST NEIGHBOR SPACING =====")

display(df_diff)